# 04 Results: final test evaluation and impact

- What it does: refits every model on train + validation with the frozen settings, scores the test years (2024 onward) once, and translates the results into impact for ISD.
- Inputs: `data/processed/model_table.parquet`, `data/processed/inspections_full.parquet`, `docs/FROZEN_CONFIG.json`.
- Outputs: `preds/*_test.csv`, `results/results.csv`, `results/model_differences.csv`, `results/results_by_year.csv`, `results/rule_sensitivity.csv`, `results/robustness.csv`, `results/impact.csv`, `results/priority_list.csv`, and the figures `gains_curve_test.png`, `hit_rate_bar.png`, `feature_importance.png`, `impact_bar.png` in `results/figures/`.

No setting is tuned here. Every setting comes from `docs/FROZEN_CONFIG.json`, which was committed before the test years were scored.

In [ ]:
import sys
sys.path.insert(0, "..")

import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

from src import evaluate as E
from src import features as F
from src import impact as I
from src import models as M
from src import text as T

SEED = 42
np.random.seed(SEED)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)

FROZEN = json.load(open("../docs/FROZEN_CONFIG.json"))
FINAL = FROZEN["final_model"]          # the model we would deploy, chosen on validation
assert FROZEN["history_features"] == M.HISTORY_FEATURES, "feature list changed after the freeze"
print("final model (chosen on validation):", FINAL)
print("M1 settings:", FROZEN["M1_lgbm"])
print("M2 settings:", FROZEN["M2_lgbm_text"])

## 1. Refit on train + validation, score the test years once

With the settings frozen, the validation year has done its job, so it joins the training data. LightGBM uses the number of trees that early stopping found on validation.

In [ ]:
table = M.load_model_table("../data/processed/model_table.parquet")
is_fit = table["split"].isin(["train", "val"]).to_numpy()
is_test = (table["split"] == "test").to_numpy()
has_text = (table["prior_comments"].str.strip() != "").to_numpy()

# Text features for M2. The PCA is refitted on train + validation rows only.
embeddings = T.embed_cached(table["inspection_id"], table["prior_comments"], cache_dir="../data/processed")
pca = T.fit_pca(embeddings, fit_mask=is_fit & has_text)
table = T.add_text_features(table, embeddings, pca)
FEATURES_M2 = M.HISTORY_FEATURES + T.TEXT_FEATURES

fit, test = table[is_fit], table[is_test]
# Nothing is ever fitted on the test period: not the models, not the PCA, not B1's fallback.
assert fit["date"].max() < pd.Timestamp("2024-01-01") <= test["date"].min()
assert table.loc[is_fit & has_text, "date"].max() < pd.Timestamp("2024-01-01")     # rows the PCA was fitted on
y_test = test[M.TARGET].to_numpy()
print(f"fit on {len(fit):,} inspections (2016 to 2023) | test on {len(test):,} inspections "
      f"({test['date'].min().date()} to {test['date'].max().date()}), fail rate {y_test.mean():.1%}")

In [ ]:
lgbm_keys = ["num_leaves", "learning_rate", "n_estimators"]
params_m1 = {k: FROZEN["M1_lgbm"][k] for k in lgbm_keys}
params_m2 = {k: FROZEN["M2_lgbm_text"][k] for k in lgbm_keys}

logreg = M.fit_logreg(fit)
m1 = M.fit_lgbm(fit, M.HISTORY_FEATURES, params_m1)
m2 = M.fit_lgbm(fit, FEATURES_M2, params_m2)

test_scores = {
    "B0_random": M.score_random(test),
    "B1_past_fail_rate": M.score_past_fail_rate(test, fallback=M.past_fail_rate_fallback(fit)),
    "B2_logreg": logreg.predict_proba(M.linear_matrix(test))[:, 1],
    "M1_lgbm": m1.predict_proba(test[M.HISTORY_FEATURES])[:, 1],
    "M2_lgbm_text": m2.predict_proba(test[FEATURES_M2])[:, 1],
}
for name, score in test_scores.items():
    M.save_preds(test, score, f"../preds/{name}_test.csv")

## 2. Test results

Capacity is 20% of inspections. The hit rate (`precision_at_k`) is the share of those visits that find a failure. Intervals are 95% bootstrap intervals (500 resamples of whole restaurants). B0 is the average over every random order, so its hit rate equals the fail rate.

In [ ]:
test_groups = test["licenseno"].to_numpy()      # confidence intervals resample whole restaurants
results = E.results_table(y_test, test_scores, groups=test_groups)
results.to_csv("../results/results.csv", index=False)
results.round(3)

Is the final model better than each alternative? Each row is "final model minus the other model" on the same 10,000 bootstrap resamples of whole restaurants. The verdict has three levels, defined in the frozen config before the test years were scored: "beyond noise", "borderline" (the interval's end nearest zero is about zero) and "within noise".

In [ ]:
differences = E.difference_table(y_test, test_scores, FINAL, [m for m in test_scores if m != FINAL], groups=test_groups)
differences.to_csv("../results/model_differences.csv", index=False)
hit_diffs = differences[differences["metric"] == "precision_at_k"]
print(hit_diffs.round(4).to_string(index=False))

print()
for row in hit_diffs.itertuples():
    print(f"{E.MODEL_LABELS[FINAL]} vs. {E.MODEL_LABELS[row.minus]}: {row.difference * 100:+.1f} points of hit rate "
          f"(95% CI {row.ci_lo * 100:+.1f} to {row.ci_hi * 100:+.1f}): {row.verdict}.")

### By year

The headline numbers rank all test inspections together. Here each test year is ranked on its own (top 20% of that year), to check that the result does not come from one year. 2026 stops on November 10.

In [ ]:
by_year = []
for year, idx in test.groupby("year").indices.items():
    for name, score in test_scores.items():
        by_year.append({"year": year, "model": name, "n": len(idx), "fail_rate": y_test[idx].mean(),
                        "precision_at_k": E.evaluate(y_test[idx], score[idx])["precision_at_k"]})
by_year = pd.DataFrame(by_year)
by_year.to_csv("../results/results_by_year.csv", index=False)
by_year.pivot(index="model", columns="year", values="precision_at_k").loc[list(test_scores)].round(3)

### Sensitivity: a second simple rule

The gain over "the rule" depends on the rule. B1 ranks by past failure rate, as the project brief defines it. An inspector could also rank by the number of violations at the last routine inspection. This row is not one of the five models. It shows how much of the model's performance one well-chosen number already delivers.

In [ ]:
simple_rule = M.score_simple_rule(test)
sensitivity = E.results_table(y_test, {"S1_last_routine_violations": simple_rule}, groups=test_groups)
d, lo, hi = E.paired_differences(y_test, test_scores[FINAL], simple_rule, groups=test_groups)["precision_at_k"]
sensitivity["final_model_minus_rule"], sensitivity["diff_ci_lo"], sensitivity["diff_ci_hi"] = d, lo, hi
sensitivity.to_csv("../results/rule_sensitivity.csv", index=False)
print(f"'Most violations at the last routine inspection first': hit rate {sensitivity['precision_at_k'][0]:.1%}")
print(f"{E.MODEL_LABELS[FINAL]} minus this rule: {d * 100:+.1f} points (95% CI {lo * 100:+.1f} to {hi * 100:+.1f})")

### Figures

In [ ]:
fig = E.plot_gains(y_test, test_scores, "Riskiest first: failures found vs. inspections made (2024 to 2026)",
                   path="../results/figures/gains_curve_test.png")
plt.show()

In [ ]:
fig = E.plot_hit_rate_bar(results, "Visiting the riskiest 20% first: how often a visit finds a failure",
                          path="../results/figures/hit_rate_bar.png", base_rate=y_test.mean())
plt.show()

In [ ]:
final_model = {"M1_lgbm": m1, "M2_lgbm_text": m2}[FINAL]
final_features = {"M1_lgbm": M.HISTORY_FEATURES, "M2_lgbm_text": FEATURES_M2}[FINAL]
importance = M.importance_table(final_model, final_features, top=15)
importance.to_csv("../results/feature_importance.csv", index=False)
fig = E.plot_importance(importance, "What drives the risk score",
                        path="../results/figures/feature_importance.png")
plt.show()

### Robustness: without likely late follow-ups

The 30-day rule removes most re-inspections, but a few follow-ups come later and stay in the data. They are easy to predict (they usually pass). Here we drop every test inspection whose previous visit failed and recompute the metrics with the same predictions. If the ranking of the models holds, the results do not depend on those rows.

In [ ]:
keep = (test["last_fail"] != 1).to_numpy()
robust = E.results_table(y_test[keep], {name: s[keep] for name, s in test_scores.items()}, groups=test_groups[keep])
robust.to_csv("../results/robustness.csv", index=False)
print(f"dropped {int((~keep).sum())} of {len(keep):,} test inspections whose previous visit failed")
robust[["model", "auc", "catch_at_k", "precision_at_k", "precision_at_k_lo", "precision_at_k_hi", "n"]].round(3)

## 3. Impact

All numbers come from the test period. Capacity stays fixed at 20% of routine inspections.

- Random order stands for today's routine schedule. Averaged over every possible random order, its hit rate equals the overall fail rate.
- No dollar figures: `COST_PER_FAILURE_LOW` and `COST_PER_FAILURE_HIGH` in `src/impact.py` are `None` until a sourced cost per missed failure is available.

In [ ]:
impact = I.impact_table(y_test, test["date"], test_scores, best=FINAL)

# Time to discovery: keep ISD's actual visit slots, but within each calendar quarter
# hand them out in risk order. Assumes the outcome does not depend on the week of the visit.
timing = I.weeks_earlier(test["date"], y_test, test_scores[FINAL], window="Q")
impact.loc[len(impact)] = ("Weeks earlier a failing restaurant is visited, on average", timing["mean_weeks_earlier_failures"],
                           "weeks", f"risk order within each of {timing['n_windows']} calendar quarters, actual visit slots kept; "
                                    f"{timing['n_failures']:,} failed inspections")
impact.to_csv("../results/impact.csv", index=False)
impact.assign(value=impact["value"].round(3))

In [ ]:
fig = I.plot_impact_bar(impact, E.MODEL_LABELS[FINAL].replace(", ", ",\n"),
                        "Same inspectors, more failures found", path="../results/figures/impact_bar.png")
plt.show()

## 4. Priority list for the next round of inspections

The final model is refitted on all labeled data (2016 to the snapshot date). Each active restaurant then gets a risk score for its next inspection, as of the snapshot date. "Active" means the license is active and the restaurant was inspected in the last two years. Restaurants that failed in the last 30 days are left out, because ISD re-inspects them anyway.

The drivers are the three features that push a restaurant's score up the most. The list contains business names and no owner names.

In [ ]:
full = pd.read_parquet("../data/processed/inspections_full.parquet")
as_of = pd.Timestamp(json.load(open("../data/raw/pull_metadata.json"))["pulled_at_utc"][:10])
nxt = I.next_inspection_rows(full, as_of, F.add_history_features)

# Same dtypes and categories as the training table.
nxt[M.NUM_FEATURES] = nxt[M.NUM_FEATURES].astype("float64")
for col in M.CAT_FEATURES:
    nxt[col] = pd.Categorical(nxt[col], categories=table[col].cat.categories)

if FINAL == "M2_lgbm_text":
    emb_all = T.embed_cached(table["inspection_id"], table["prior_comments"], cache_dir="../data/processed")
    pca_all = T.fit_pca(emb_all, fit_mask=has_text)
    table_all = T.add_text_features(table.drop(columns=T.TEXT_FEATURES), emb_all, pca_all)
    emb_next = T.embed_cached(nxt["inspection_id"], nxt["prior_comments"], cache_dir="../data/processed",
                              name="embeddings_next")
    nxt = T.add_text_features(nxt, emb_next, pca_all)
else:
    table_all = table

deploy_params = {k: FROZEN[FINAL][k] for k in lgbm_keys}
deploy_model = M.fit_lgbm(table_all, final_features, deploy_params)
nxt["risk"] = deploy_model.predict_proba(nxt[final_features])[:, 1]
nxt["drivers"] = I.top_drivers(deploy_model, nxt[final_features], nxt, text_features=T.TEXT_FEATURES)

priority = (nxt.sort_values("risk", ascending=False)
            .assign(rank=lambda d: np.arange(1, len(d) + 1), risk_pct=lambda d: (d["risk"] * 100).round(0).astype(int))
            [["rank", "businessname", "licenseno", "neighborhood", "risk_pct", "drivers"]]
            .rename(columns={"businessname": "business_name", "licenseno": "license_no", "drivers": "top_3_drivers"}))
assert not F.OWNER_COLUMNS & set(priority.columns)
priority.to_csv("../results/priority_list.csv", index=False)
print(f"{len(priority):,} active restaurants scored as of {as_of.date()}")
priority.head(15)